# ANTON Antenna Reproduction: Triangular Patch Antenna
**Automated PyAEDT / ANSYS HFSS Model Generation**

---

### Project Overview
- **Project Name:** `Triangular Patch Antenna`
- **Project ID:** `7570b7b9-7b48-4fa5-888f-b6434cca6b50`
- **Description:** Universal microstrip antenna geometry exported from ANTON
- **Internal Units:** SI Metres ($m$)
- **Operating Center Frequency:** `2.4 GHz`
- **Simulation Frequency Sweep:** `1.7999999999999998 - 3.0 GHz` (201 points)
- **Number of Physical Layers:** 3
- **Number of Geometry Objects:** 4

---

### Standalone Execution Instructions
> **ANTON Architectural Principle:**
> ANTON operates completely independently of ANSYS HFSS. This notebook is a **self-contained script** intended to be executed in your local Jupyter environment.
>
> 1. Ensure you have **ANSYS Electronics Desktop (AEDT)** installed on your machine (versions 2022.1 - 2025.1 supported).
> 2. Ensure your Python environment has `pyaedt`, `numpy`, and `matplotlib`:
>    ```bash
>    pip install pyaedt numpy matplotlib
>    ```
> 3. Execute the cells sequentially from top to bottom.
> 4. Inspect the recreated 3D CAD model directly inside the ANSYS HFSS graphical interface.
> 5. Review the manual solve cell, run the electromagnetic analysis, and extract S-parameters, VSWR, and radiation patterns.


## 1. Environment & Dependency Check
Verifies Python dependencies, PyAEDT availability, and system compatibility.

In [ ]:
# Check required packages
import sys

print(f"Python interpreter: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

try:
    import pyaedt
    print(f"PyAEDT version: {pyaedt.__version__}")
except ImportError:
    print("WARNING: 'pyaedt' is not installed in this environment.")
    print("Please install PyAEDT to connect to ANSYS HFSS: pip install pyaedt")

try:
    import numpy as np
    print(f"NumPy version: {np.__version__}")
except ImportError:
    print("WARNING: 'numpy' is not installed: pip install numpy")

try:
    import matplotlib
    print(f"Matplotlib version: {matplotlib.__version__}")
except ImportError:
    print("WARNING: 'matplotlib' is not installed: pip install matplotlib")


## 2. Launch / Connect to ANSYS HFSS Desktop
Connects to an existing HFSS session or launches a new AEDT desktop session using PyAEDT.

In [ ]:
import os
from pathlib import Path

try:
    from pyaedt import Hfss
except ImportError:
    Hfss = None
    raise RuntimeError(
        "PyAEDT ('pyaedt') is not installed in this Python environment. "
        "Please install it before running this reproduction notebook: pip install pyaedt"
    )

design_name = "Triangular Patch Antenna"
specified_version = None
non_graphical = False
new_desktop_session = False

app = None
try:
    print(f"Connecting to ANSYS HFSS for design '{design_name}'...")
    app = Hfss(
        projectname=None,
        designname=design_name,
        solution_type="Terminal",
        specified_version=specified_version,
        non_graphical=non_graphical,
        new_desktop_session=new_desktop_session,
    )
    if app is None:
        raise RuntimeError("Failed to obtain Hfss application instance from PyAEDT.")
    # Enforce SI Metres in the HFSS Modeler
    app.modeler.model_units = "meter"
    print("Successfully connected to ANSYS HFSS Desktop.")
    print(f"Active Project: {app.project_name} | Active Design: {app.design_name}")
    print(f"HFSS Modeler units set to: {app.modeler.model_units}")
except Exception as e:
    raise RuntimeError(
        f"Critical failure: Could not connect to live ANSYS HFSS instance: {e}. "
        "Ensure ANSYS Electronics Desktop is running or valid AEDT licenses are configured."
    ) from e


## 3. Material Definitions & HFSS Library Synchronization
Synchronizes substrate dielectrics, ground/patch conductors, and loss tangents with the HFSS material database.

In [ ]:
# ===========================================================================
# 2. MATERIAL DEFINITIONS & HFSS LIBRARY SYNCHRONIZATION
# ===========================================================================
print('Configuring materials in HFSS library...')
material_name_map = {}  # Maps ANTON material ID to HFSS material name

# Material: 'Rogers RO4350B' (Category: dielectric)
mat_name = 'Rogers_RO4350B'
if app is not None and not app.materials.check_if_exists(mat_name):
    new_mat = app.materials.add_material(mat_name)
    new_mat.permittivity = 3.66
    new_mat.dielectric_loss_tangent = 0.0037
    print(f"Created custom dielectric material: '{mat_name}' (eps_r=3.66, tan_d=0.0037)")
material_name_map['973f3c70-f5af-42dd-88aa-95c069766009'] = mat_name

# Material: 'Copper' (Category: conductor)
if app is not None and app.materials.check_if_exists('copper'):
    material_name_map['e8666a96-2d40-456d-8dbb-882e9641f679'] = 'copper'
    print(f"Using built-in HFSS material: 'copper'")
else:
    mat_name = 'Copper'
    if app is not None and not app.materials.check_if_exists(mat_name):
        new_mat = app.materials.add_material(mat_name)
        new_mat.conductivity = 58000000.0
        print(f"Created custom conductor material: '{mat_name}' (sigma=58000000.0 S/m)")
    material_name_map['e8666a96-2d40-456d-8dbb-882e9641f679'] = mat_name

print(f'Successfully synchronized {len(material_name_map)} material(s) in HFSS.')

## 4. Physical Stackup: Substrate & Ground Plane Modeling
Creates the 3D dielectric substrate block and ground plane in HFSS with exact Z-stackup coordinates.

In [ ]:
# ===========================================================================
# 3. DIELECTRIC SUBSTRATE & GROUND PLANE CONSTRUCTION
# ===========================================================================
print('Constructing physical layer stackup in HFSS modeler...')

# Collect created modeler objects for verification
hfss_objects = {}

# Layer: 'Ground' (Type: ground, z=-0.0350 mm, t=0.0350 mm)
# Create rectangular solid for Ground
box_Ground = app.modeler.create_box(origin=[-0.03, -0.03, -3.5e-05], sizes=[0.06, 0.06, 3.5e-05], name='Ground', matname=material_name_map.get('e8666a96-2d40-456d-8dbb-882e9641f679', 'vacuum'))
hfss_objects['13bc2c71-0463-433b-9360-d9393a55e9f9'] = box_Ground.name
print(f"  Created box solid: '{box_Ground.name}' for layer 'Ground'")

# Layer: 'Substrate' (Type: substrate, z=0.0000 mm, t=0.7620 mm)
# Create rectangular solid for Substrate
box_Substrate = app.modeler.create_box(origin=[-0.03, -0.03, 0.0], sizes=[0.06, 0.06, 0.000762], name='Substrate', matname=material_name_map.get('973f3c70-f5af-42dd-88aa-95c069766009', 'vacuum'))
hfss_objects['8e599ada-1e98-4970-8f0b-4d84d4aa9908'] = box_Substrate.name
print(f"  Created box solid: '{box_Substrate.name}' for layer 'Substrate'")

print('Dielectric substrate and ground structure created successfully.')

## 5. Radiating Patch, Feed Line & CSG Cutouts Modeling
Generates planar conductors, polygons, circles, rings, slots, and performs CSG subtractions in the HFSS modeler.

In [ ]:
# ===========================================================================
# 4. RADIATING PATCH, FEED, AND CSG SLOTS/CUTOUTS RECONSTRUCTION
# ===========================================================================
print('Constructing radiating patch elements and feed geometry in HFSS...')

if app is not None:
    # --- Layer: 'Patch' (patch) at z=0.7620 mm ---
    # Triangular patch solid: TriangularRadiator
    pts_TriangularRadiator = [[-0.0175, -0.01010362971081845, 0.000762], [0.0175, -0.01010362971081845, 0.000762], [0.0, 0.0202072594216369, 0.000762]]
    solid_TriangularRadiator = app.modeler.create_polyline(points=pts_TriangularRadiator, close_surface=True, cover_surface=True, name='TriangularRadiator', matname=material_name_map.get('e8666a96-2d40-456d-8dbb-882e9641f679', 'copper'))
    app.modeler.extrude_sheet(solid_TriangularRadiator.name, 3.5e-05)
    hfss_objects['8bf6411b-ecc8-4f46-a5a2-35c9d8af4b99'] = solid_TriangularRadiator.name
    app.modeler.set_color(solid_TriangularRadiator.name, [245, 158, 11])
    print(f"  Created triangular patch: '{solid_TriangularRadiator.name}'")
    # Rectangular conductor: TriangleFeedLine
    solid_TriangleFeedLine = app.modeler.create_box(origin=[-0.00085, -0.03, 0.000762], sizes=[0.0017, 0.01989637028918155, 3.5e-05], name='TriangleFeedLine', matname=material_name_map.get('e8666a96-2d40-456d-8dbb-882e9641f679', 'copper'))
    hfss_objects['fa9a1dc8-9f38-49c3-a263-e7849f02978d'] = solid_TriangleFeedLine.name
    app.modeler.set_color(solid_TriangleFeedLine.name, [245, 158, 11])
    print(f"  Created rectangular patch: '{solid_TriangleFeedLine.name}'")

    print('Radiating patch and feed geometry constructed successfully.')
else:
    print('Notice: app is None (offline mode). Modeler patch generation skipped.')

## 6. Feed & Excitation Port Configuration
Constructs port excitation sheets (Lumped Port, Wave Port, or Coaxial Probe) referencing the ground plane.

In [ ]:
# ===========================================================================
# 5. PORT & EXCITATION CONFIGURATION
# ===========================================================================
print('Configuring RF excitation ports in HFSS...')

# Identify ground reference conductor in HFSS modeler
ground_names = [obj.name for obj in app.modeler.object_list if 'ground' in obj.name.lower()]
if not ground_names:
    raise RuntimeError('CRITICAL: No ground plane conductor found in HFSS modeler for port reference.')
ref_ground = ground_names[0]
print(f"  Reference ground conductor: '{ref_ground}'")

# Verified feed conductor: 'TriangleFeedLine' (signal trace)
# Port location: x=0.000 mm, y=-30.000 mm, spanning z=[0.000, 0.762] mm
port_w = 0.0017
port_h = 0.000762
port_x0 = -0.00085
port_y = -0.03
port_z0 = 0.0

# Construct 2D excitation port sheet in XZ plane bridging ground to signal trace
try:
    port_sheet = app.modeler.create_rectangle(
        orientation='XZ',
        origin=[port_x0, port_y, port_z0],
        sizes=[port_w, port_h],
        name='Port1_Sheet',
    )
except Exception:
    port_sheet = app.modeler.create_rectangle(
        csPlane='XZ',
        position=[port_x0, port_y, port_z0],
        dimension_list=[port_w, port_h],
        name='Port1_Sheet',
    )

if port_sheet is None:
    raise RuntimeError('CRITICAL: Failed to construct Port1_Sheet in HFSS modeler.')
print(f"  Created excitation sheet: '{port_sheet.name}'")

# Integration line: directed +Z from reference ground to signal trace
int_line_start = [0.0, -0.03, 0.0]
int_line_stop = [0.0, -0.03, 0.000762]

# Assign 50-Ohm Lumped Port with integration line
try:
    app.lumped_port(
        assignment=port_sheet.name,
        reference=ref_ground,
        integration_line=[int_line_start, int_line_stop],
        create_port_sheet=False,
        port_name='1',
    )
    print(f"  Assigned Lumped Port '1' with +Z integration line to '{port_sheet.name}'")
except Exception as e1:
    try:
        app.create_lumped_port_to_sheet(
            sheet_name=port_sheet.name,
            reference_object_list=[ref_ground],
            integration_line=[int_line_start, int_line_stop],
            portname='1',
        )
        print(f"  Assigned Lumped Port '1' via create_lumped_port_to_sheet to '{port_sheet.name}'")
    except Exception as e2:
        raise RuntimeError(f"CRITICAL: Failed to assign Lumped Port '1' in HFSS: {e1} / {e2}")

print('Excitation port configuration successfully completed.')

## 7. Airbox Enclosure & Radiation Boundary
Encloses the antenna structure in an airbox padded by $\lambda_0 / 4$ at 2.4 GHz with an ABC/PML radiation boundary.

In [ ]:
# ===========================================================================
# 6. AIRBOX ENCLOSURE & RADIATION BOUNDARY CONFIGURATION
# ===========================================================================
print('Creating airbox with quarter-wave radiation padding at 2.4 GHz...')

center_freq_ghz = 2.4
lambda_quarter = 0.031228  # lambda_0 / 4 in metres (31.23 mm)

# Verified canonical geometry bounds from ANTON project (metres)
canon_xmin, canon_ymin, canon_zmin = -0.030000, -0.030000, -0.000035
canon_xmax, canon_ymax, canon_zmax = 0.030000, 0.030000, 0.000797

# Query model bounding box from HFSS modeler with fallback to canonical project bounds
try:
    bb = app.modeler.get_bounding_box()
    if bb and len(bb) >= 6:
        xmin, ymin, zmin = float(bb[0]), float(bb[1]), float(bb[2])
        xmax, ymax, zmax = float(bb[3]), float(bb[4]), float(bb[5])
    else:
        xmin, ymin, zmin = canon_xmin, canon_ymin, canon_zmin
        xmax, ymax, zmax = canon_xmax, canon_ymax, canon_zmax
except Exception as err_bb:
    print(f'Warning: Could not query modeler bounding box ({err_bb}); using verified canonical project bounds.')
    xmin, ymin, zmin = canon_xmin, canon_ymin, canon_zmin
    xmax, ymax, zmax = canon_xmax, canon_ymax, canon_zmax

# Calculate airbox dimensions with quarter-wave padding
air_x0 = xmin - lambda_quarter
air_y0 = ymin - lambda_quarter
air_z0 = zmin - lambda_quarter
air_dx = (xmax - xmin) + 2.0 * lambda_quarter
air_dy = (ymax - ymin) + 2.0 * lambda_quarter
air_dz = (zmax - zmin) + 2.0 * lambda_quarter

# Create Airbox solid
airbox = app.modeler.create_box(
    origin=[air_x0, air_y0, air_z0],
    sizes=[air_dx, air_dy, air_dz],
    name='Airbox',
    matname='vacuum'
)
if airbox is None:
    raise RuntimeError('Failed to create Airbox solid in HFSS modeler.')

# Set airbox transparency in modeler
app.modeler.set_color(airbox.name, [186, 230, 253])
try:
    app.modeler.set_transparency(airbox.name, 0.85)
except Exception:
    pass

# Assign Radiation Boundary to Airbox outer surfaces
rad_assigned = False
try:
    app.assign_radiation_boundary_to_objects(airbox.name)
    rad_assigned = True
    print(f"  Radiation boundary successfully assigned to object: '{airbox.name}'")
except Exception as err_obj:
    try:
        app.assign_radiation_boundary_to_faces(airbox.faces)
        rad_assigned = True
        print('  Radiation boundary assigned to airbox faces')
    except Exception as err_faces:
        raise RuntimeError(
            f"Failed to assign radiation boundary to Airbox! "
            f"Object method error: {err_obj}; Face method error: {err_faces}"
        ) from err_faces

if not rad_assigned:
    raise RuntimeError('Critical verification failure: Radiation boundary was not assigned to Airbox.')

print('Airbox and radiation boundaries successfully configured.')

## 8. Solution Setup & Frequency Sweep
Configures adaptive mesh refinement setup at 2.4 GHz and an interpolating frequency sweep (1.7999999999999998 - 3.0 GHz).

In [ ]:
# ===========================================================================
# 7. SOLUTION SETUP & FREQUENCY SWEEP CONFIGURATION
# ===========================================================================
print('Configuring solution setup at 2.4 GHz...')

if app is None:
    raise RuntimeError('Critical: app is None. No active HFSS session to configure solution setup.')

setup_name = 'Setup1'
sweep_name = 'Sweep1'

# Create HFSS solution setup with adaptive meshing
setup = None
try:
    setup = app.create_setup(name=setup_name, setup_type='HFSSDrivenAuto')
except Exception as e_auto:
    try:
        setup = app.create_setup(name=setup_name, setup_type='HFSS')
    except Exception as e_hfss:
        try:
            setup = app.create_setup(name=setup_name)
        except Exception as e_gen:
            raise RuntimeError(
                f"Failed to create HFSS solution setup '{setup_name}'! "
                f"HFSSDrivenAuto error: {e_auto}; HFSS error: {e_hfss}; Generic error: {e_gen}"
            ) from e_gen

if setup is None:
    raise RuntimeError(f"Failed to create HFSS solution setup '{setup_name}'.")

setup.props['Frequency'] = '2.4GHz'
setup.props['MaximumPasses'] = 15
setup.props['MaxDeltaS'] = 0.02
print(f"  Configured '{setup.name}': Center=2.4 GHz, MaxPasses=15, MaxDeltaS=0.02")

# Configure broadband frequency sweep
print('Configuring Interpolating frequency sweep from 1.7999999999999998 to 3.0 GHz (201 points)...')
sweep = None
try:
    sweep = setup.create_frequency_sweep(
        name=sweep_name,
        unit='GHz',
        start_frequency=1.7999999999999998,
        stop_frequency=3.0,
        num_of_freq_points=201,
        sweep_type='Interpolating',
    )
except Exception as e1:
    try:
        sweep = setup.create_frequency_sweep(
            name=sweep_name,
            unit='GHz',
            start_frequency='1.7999999999999998GHz',
            stop_frequency='3.0GHz',
            num_of_freq_points=201,
            sweep_type='Interpolating',
        )
    except Exception as e2:
        raise RuntimeError(
            f"Failed to configure frequency sweep '{sweep_name}'! "
            f"Primary method error: {e1}; Fallback method error: {e2}"
        ) from e2

if sweep is None:
    raise RuntimeError(f"Failed to create frequency sweep '{sweep_name}'.")

print(f"  Configured frequency sweep: '{sweep.name}'")

# Optional: Adaptive mesh refinement on patch conductors
print('Assigning initial mesh refinement on conductive radiator faces...')
try:
    patch_conductors = [obj.name for obj in app.modeler.object_list if 'patch' in obj.name.lower() or 'radiator' in obj.name.lower()]
    if patch_conductors:
        app.mesh.assign_length_mesh(patch_conductors, isinside=False, maxlength=0.005, name='Mesh_PatchEdge')
        print(f"  Assigned length mesh refinement to: {patch_conductors}")
except Exception as e:
    print(f'  Mesh refinement note (optional): {e}')

print('HFSS simulation setup and frequency sweep fully configured.')

## 9. Manual Simulation Solve Execution
> **ANTON Manual Execution Gate:**
> Simulations are intentionally not executed automatically. Inspect your 3D design in ANSYS HFSS Desktop first.
> When ready, uncomment and execute the cell below.

In [ ]:
# ===========================================================================
# 8. MANUAL SIMULATION EXECUTION
# ===========================================================================
# NOTE: In accordance with ANTON architectural principles, simulations are NOT
# automatically executed. Inspect the created geometry in your ANSYS HFSS desktop.
# When ready, uncomment and execute the lines below to solve:

# print('Starting manual solve of Setup1 : Sweep1 in HFSS...')
# if 'app' not in globals() or app is None:
#     raise RuntimeError('Cannot solve: No active HFSS session (app is None).')
# app.analyze_setup('Setup1')
# print('Simulation solve completed successfully.')

## 10. Result Extraction, Impedance & Far-Field Metrics Derivation
Extracts $S_{11}$, derives $\text{VSWR} = \frac{1 + |\Gamma|}{1 - |\Gamma|}$, calculates complex input impedance $Z_{\text{in}}$, and distinguishes Gain vs. Directivity vs. Realized Gain.

In [ ]:
# ===========================================================================
# 9. RESULT EXTRACTION, ANTENNA METRICS & DERIVATIONS
# ===========================================================================
import numpy as np
import matplotlib.pyplot as plt
import json
import csv
from pathlib import Path

print('Extracting simulation results from HFSS post-processor...')

# Execution status flags: NOT_SIMULATED, SIMULATION_COMPLETED, RESULTS_EXTRACTED, RESULTS_UNAVAILABLE, EXTRACTION_FAILED
simulation_status = 'NOT_SIMULATED'
results_status = 'RESULTS_UNAVAILABLE'

freqs_ghz = np.array([])
s11_db = np.array([])
s11_complex = np.array([])
vswr = np.array([])
z_in = np.array([])
r_in = np.array([])
x_in = np.array([])
mismatch_loss_factor = np.array([])
gamma_mag = np.array([])
Z0 = 50.0

if 'app' not in globals() or app is None:
    simulation_status = 'NOT_SIMULATED'
    results_status = 'RESULTS_UNAVAILABLE'
    print('Status: NOT_SIMULATED / RESULTS_UNAVAILABLE (no active HFSS session).')
    print('To simulate: connect to HFSS, execute Cell 8 (manual solve), and re-run this cell.')
else:
    try:
        # Query solved S(1,1) frequency sweep data from HFSS post-processor
        s11_data = app.post.get_report_data('S(1,1)', setup_sweep_name='Setup1 : Sweep1')
        if s11_data is not None and hasattr(s11_data, 'sweeps') and 'Freq' in s11_data.sweeps:
            raw_freqs = np.array(s11_data.sweeps['Freq'], dtype=float)
            if len(raw_freqs) > 0:
                # Convert Hz to GHz if returned in Hz
                freqs_ghz = raw_freqs / 1e9 if np.max(raw_freqs) > 1e6 else raw_freqs
                s11_db = np.array(s11_data.data_db(), dtype=float)
                s11_complex = np.array(s11_data.data_complex(), dtype=complex)
                simulation_status = 'SIMULATION_COMPLETED'
                results_status = 'RESULTS_EXTRACTED'
                print(f'Successfully extracted {len(freqs_ghz)} solved S11 data points from HFSS.')
            else:
                results_status = 'RESULTS_UNAVAILABLE'
                print('Status: RESULTS_UNAVAILABLE (Setup1:Sweep1 contains no solved frequency points).')
        else:
            results_status = 'RESULTS_UNAVAILABLE'
            print('Status: RESULTS_UNAVAILABLE (no report data returned for Setup1:Sweep1).')
            print('Please verify that Setup1 has been solved in HFSS.')
    except Exception as e:
        results_status = 'EXTRACTION_FAILED'
        print(f'Result extraction note: {e}')
        print('Status: EXTRACTION_FAILED (Setup1:Sweep1 may not have been solved).')

# Compute antenna metrics ONLY if real HFSS results were extracted
if results_status == 'RESULTS_EXTRACTED' and len(freqs_ghz) > 0 and len(s11_db) > 0:
    # 1. Reflection Coefficient Magnitude: |Gamma| = |S11|
    gamma_mag = np.abs(s11_complex) if len(s11_complex) > 0 else 10.0 ** (s11_db / 20.0)

    # 2. Voltage Standing Wave Ratio (VSWR): (1 + |Gamma|) / (1 - |Gamma|)
    vswr = (1.0 + gamma_mag) / np.clip(1.0 - gamma_mag, 1e-4, 1.0)

    # 3. Complex Input Impedance: Z_in = Z0 * (1 + Gamma) / (1 - Gamma), where Z0 = 50 Ohm
    if len(s11_complex) > 0:
        z_in = Z0 * (1.0 + s11_complex) / (1.0 - s11_complex)
        r_in = np.real(z_in)
        x_in = np.imag(z_in)
    else:
        r_in = np.full_like(freqs_ghz, 50.0)
        x_in = np.zeros_like(freqs_ghz)

    # 4. Far-field / mismatch metrics:
    # - Directivity D: Directional power density relative to isotropic radiator (losses excluded)
    # - Gain G = eta_rad * D: Takes into account dielectric and conductor ohmic losses
    # - Realized Gain G_realized = (1 - |Gamma|^2) * G: Takes into account impedance mismatch reflection losses
    mismatch_loss_factor = np.clip(1.0 - gamma_mag ** 2, 0.0, 1.0)

    min_idx = int(np.argmin(s11_db))
    f_res = float(freqs_ghz[min_idx])
    s11_min = float(s11_db[min_idx])
    vswr_min = float(vswr[min_idx])
    print('\nExtracted & Derived Antenna Metrics (Real HFSS Data):')
    print(f'  Resonant Frequency: {f_res:.4f} GHz')
    print(f'  Minimum S11: {s11_min:.2f} dB')
    print(f'  Minimum VSWR: {vswr_min:.2f}')
    if len(r_in) > min_idx:
        print(f'  Input Impedance at resonance: {r_in[min_idx]:.2f} + j({x_in[min_idx]:.2f}) Ohm')
    print(f'  Mismatch Efficiency at resonance: {mismatch_loss_factor[min_idx]*100:.1f}%')
else:
    print(f'\nSimulation status: {simulation_status} | Results status: {results_status}')
    print('Notice: Antenna metrics cannot be computed until simulation is solved in HFSS.')

## 11. Visualization: S-Parameters, VSWR, Impedance & Far-Field Radiation Patterns
Generates 2D Cartesian and polar plots for return loss, VSWR, complex input impedance, and elevation radiation cuts.

In [ ]:
# ===========================================================================
# 10. VISUALIZATION: S11, VSWR, IMPEDANCE & RADIATION PATTERNS
# ===========================================================================
if results_status != 'RESULTS_EXTRACTED' or len(s11_db) == 0:
    print('=' * 75)
    print('VISUALIZATION SKIPPED: No simulation results available.')
    print(f'Status: {results_status} ({simulation_status})')
    print('To display plots, solve Setup1 in HFSS (Cell 8) and re-run Cell 9.')
    print('=' * 75)
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Plot 1: Return Loss |S11| (dB)
    ax1 = axes[0]
    ax1.plot(freqs_ghz, s11_db, 'b-', linewidth=2, label='|S11| (dB)')
    ax1.axhline(-10.0, color='r', linestyle='--', alpha=0.7, label='-10 dB threshold')
    ax1.set_xlabel('Frequency (GHz)', fontsize=10)
    ax1.set_ylabel('Return Loss |S11| (dB)', fontsize=10)
    ax1.set_title('S11 vs Frequency', fontsize=11, fontweight='bold')
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend(loc='upper right')

    # Plot 2: VSWR
    ax2 = axes[1]
    ax2.plot(freqs_ghz, vswr, 'g-', linewidth=2, label='VSWR')
    ax2.axhline(2.0, color='r', linestyle='--', alpha=0.7, label='VSWR = 2.0 reference')
    ax2.set_xlabel('Frequency (GHz)', fontsize=10)
    ax2.set_ylabel('VSWR', fontsize=10)
    ax2.set_ylim(1.0, 6.0)
    ax2.set_title('Voltage Standing Wave Ratio (VSWR)', fontsize=11, fontweight='bold')
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend(loc='upper right')

    # Plot 3: Input Impedance (Real & Imaginary)
    ax3 = axes[2]
    if len(r_in) > 0 and len(x_in) > 0:
        ax3.plot(freqs_ghz, r_in, 'r-', linewidth=1.8, label='Resistance R_in (Ohm)')
        ax3.plot(freqs_ghz, x_in, 'm--', linewidth=1.8, label='Reactance X_in (Ohm)')
        ax3.axhline(50.0, color='k', linestyle=':', alpha=0.5, label='50 Ohm Target')
        ax3.axhline(0.0, color='k', linestyle='-', alpha=0.3)
        ax3.set_xlabel('Frequency (GHz)', fontsize=10)
        ax3.set_ylabel('Impedance (Ohm)', fontsize=10)
        ax3.set_title('Input Impedance (R_in + j X_in)', fontsize=11, fontweight='bold')
        ax3.grid(True, linestyle='--', alpha=0.6)
        ax3.legend(loc='upper right')

    fig.tight_layout()
    plt.show()

    # Far-field radiation pattern query (real HFSS data only, no synthetic placeholders)
    try:
        ff_data = app.post.get_report_data('dB(GainTotal)', setup_sweep_name='Setup1 : LastAdaptive', context='3D')
        if ff_data:
            print('Far-field pattern data available from HFSS post-processor.')
        else:
            print('Notice: Far-field radiation pattern report not generated or unavailable.')
    except Exception:
        print('Notice: Far-field radiation pattern extraction requires solved 3D far-field setup in HFSS.')

## 12. Multi-Format Simulation Result Export
Exports simulation results and antenna performance summaries to:
1. `s_parameters.csv` (CSV format)
2. `simulation_summary.json` (Structured JSON summary)
3. `antenna_s11.s1p` (Standard Touchstone 1-Port format)

In [ ]:
# ===========================================================================
# 11. MULTI-FORMAT DATA EXPORT: CSV, JSON & TOUCHSTONE (.S1P)
# ===========================================================================
if results_status != 'RESULTS_EXTRACTED' or len(s11_db) == 0:
    print('=' * 75)
    print('DATA EXPORT SKIPPED: No simulation results available to export.')
    print(f'Status: {results_status} ({simulation_status})')
    print('Solve Setup1 in HFSS (Cell 8) and re-run Cell 9 before exporting CSV, JSON, and Touchstone.')
    print('=' * 75)
else:
    output_dir = Path('./simulation_outputs')
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Export S-Parameters to CSV
    csv_path = output_dir / 's_parameters.csv'
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Frequency_GHz', 'S11_dB', 'S11_Real', 'S11_Imag', 'VSWR', 'R_in_Ohm', 'X_in_Ohm'])
        for i in range(len(freqs_ghz)):
            re_val = float(np.real(s11_complex[i])) if len(s11_complex) > i else 0.0
            im_val = float(np.imag(s11_complex[i])) if len(s11_complex) > i else 0.0
            writer.writerow([
                round(float(freqs_ghz[i]), 6),
                round(float(s11_db[i]), 4),
                round(re_val, 6),
                round(im_val, 6),
                round(float(vswr[i]), 4),
                round(float(r_in[i]), 4),
                round(float(x_in[i]), 4),
            ])
    print(f"Exported S-parameters CSV: '{csv_path}'")

    # 2. Export Structured Summary to JSON
    json_path = output_dir / 'simulation_summary.json'
    summary = {
        'design_name': 'Triangular Patch Antenna',
        'simulation_status': simulation_status,
        'results_status': results_status,
        'num_frequency_points': int(len(freqs_ghz)),
        'frequency_range_ghz': [float(np.min(freqs_ghz)), float(np.max(freqs_ghz))],
        'resonance_frequency_ghz': float(freqs_ghz[np.argmin(s11_db)]) if len(s11_db) > 0 else None,
        'minimum_s11_db': float(np.min(s11_db)) if len(s11_db) > 0 else None,
        'minimum_vswr': float(np.min(vswr)) if len(vswr) > 0 else None,
        'impedance_at_resonance': {
            'R_Ohm': float(r_in[np.argmin(s11_db)]) if len(s11_db) > 0 else None,
            'X_Ohm': float(x_in[np.argmin(s11_db)]) if len(s11_db) > 0 else None,
        },
        'units': {'frequency': 'GHz', 'return_loss': 'dB', 'impedance': 'Ohm'},
    }
    json_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f"Exported Simulation Summary JSON: '{json_path}'")

    # 3. Export Standard Touchstone 1-Port (.s1p) Format
    s1p_path = output_dir / 'antenna_s11.s1p'
    with open(s1p_path, 'w', encoding='utf-8') as f:
        f.write('! Standard Touchstone 1-Port File generated by ANTON\n')
        f.write('! Format: Frequency (GHz) | S11 Real | S11 Imaginary | Reference Impedance: 50 Ohm\n')
        f.write('# GHz S RI R 50.0\n')
        for i in range(len(freqs_ghz)):
            re_val = float(np.real(s11_complex[i])) if len(s11_complex) > i else 0.0
            im_val = float(np.imag(s11_complex[i])) if len(s11_complex) > i else 0.0
            f.write(f'{freqs_ghz[i]:.6f} {re_val:.8f} {im_val:.8f}\n')
    print(f"Exported Standard Touchstone S1P: '{s1p_path}'")
    print('All simulation results exported successfully.')

## 13. Project Save & Clean Session Release
Saves the ANSYS HFSS project file and cleanly releases the PyAEDT desktop session handle.

In [ ]:
# Save project and release desktop handle
if app is not None:
    try:
        app.save_project()
        print(f"HFSS Project '{app.project_name}' saved successfully.")
        # Release desktop handle without forcibly closing the user's GUI session
        app.release_desktop(close_projects=False, close_desktop=False)
        print("Released PyAEDT desktop handle cleanly.")
    except Exception as e:
        print(f"Project save note: {e}")
else:
    print("Offline mode: No active HFSS session to save.")
